In [ ]:
!pip install langchain-openai
!pip install python-dotenv 
!pip install langchain-teddynote

In [2]:
import os
import base64
from langchain_openai import ChatOpenAI
from langchain_teddynote.models import MultiModal
from langchain_teddynote.messages import stream_response
from dotenv import load_dotenv
from pathlib import Path
from langchain_core.messages import HumanMessage, SystemMessage

load_dotenv()

True

In [ ]:
llm = ChatOpenAI(
    temperature=0.1,
    model="gpt-4o-mini"
)


def image_url(image_source):
    """Convert a local image path to a data URL; keep HTTP URLs unchanged."""
    if str(image_source).startswith(("http://", "https://")):
        return str(image_source)

    image_path = Path(image_source)
    mime_type = "image/png" if image_path.suffix.lower() == ".png" else "image/jpeg"
    encoded_image = base64.b64encode(image_path.read_bytes()).decode("utf-8")
    return f"data:{mime_type};base64,{encoded_image}"


def stream_image_response(image_source, user_prompt, system_prompt=None):
    content = [
        {"type": "text", "text": user_prompt},
        {"type": "image_url", "image_url": {"url": image_url(image_source)}},
    ]
    messages = [HumanMessage(content=content)]
    if system_prompt:
        messages.insert(0, SystemMessage(content=system_prompt))

    for token in llm.stream(messages):
        print(token.content, end="", flush=True)
    print()


multimodal_llm = MultiModal(llm)
IMAGE_URL = "https://images.unsplash.com/photo-1470770841072-f978cf4d019e?auto=format&fit=crop&w=1200&q=80"


# 이미지 파일로 부터 질의
answer = multimodal_llm.stream(IMAGE_URL,user_prompt="이 사진에 무엇이 있는지 한국어로 설명해줘.")
stream_response(answer)